# Détection de fraude par carte bancaire — Exploration & Modélisation

**Dataset** : [Credit Card Fraud Detection (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

284 807 transactions de cartes bancaires européennes réalisées en septembre 2013, dont 492 fraudes (~0,172%). Les features `V1` à `V28` sont issues d'une transformation PCA (anonymisation) ; seules `Time`, `Amount` et `Class` (la cible) sont en clair.

**Objectif** : construire un modèle qui détecte les transactions frauduleuses, en gérant correctement le fort déséquilibre de classes.

**Plan** :
1. Chargement & aperçu
2. Analyse exploratoire (déséquilibre, distributions, corrélations)
3. Préparation des données (split stratifié, scaling)
4. Modèle de référence (régression logistique)
5. Comparaison de modèles (Random Forest, gestion du déséquilibre)
6. Choix du seuil de décision
7. Sauvegarde du modèle final


## 1. Chargement & aperçu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_auc_score, average_precision_score,
    f1_score, RocCurveDisplay, PrecisionRecallDisplay
)
import joblib

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
# Remplacez par le chemin de votre fichier réel une fois téléchargé depuis Kaggle
DATA_PATH = "../data/creditcard.csv"

df = pd.read_csv(DATA_PATH)
print("Dimensions :", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Valeurs manquantes (ce dataset n'en a normalement aucune, mais on vérifie par principe)
df.isna().sum().sum()

## 2. Analyse exploratoire

### 2.1 Déséquilibre des classes

C'est LE point central de ce dataset : la classe minoritaire (fraude) représente une fraction infime des observations. Toute décision de modélisation qui suit doit en tenir compte.

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print(class_counts)
print()
print(class_pct.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="Class", data=df, ax=axes[0])
axes[0].set_title("Nombre de transactions par classe")
axes[0].set_xticklabels(["Normale (0)", "Fraude (1)"])

axes[1].pie(class_counts, labels=["Normale", "Fraude"], autopct="%1.3f%%", colors=["#4C72B0", "#C44E52"])
axes[1].set_title("Proportion")
plt.tight_layout()
plt.show()

**À retenir** : avec un tel déséquilibre, un modèle qui prédit systématiquement "pas de fraude" obtiendrait déjà >99% d'accuracy. **L'accuracy est donc une métrique inutile ici** — on utilisera precision, recall, F1 et PR-AUC pour évaluer les modèles.

### 2.2 Distribution de `Amount` et `Time`

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(df["Amount"], bins=50, ax=axes[0, 0])
axes[0, 0].set_title("Distribution de Amount (toutes transactions)")

sns.histplot(df[df["Amount"] < 500]["Amount"], bins=50, ax=axes[0, 1])
axes[0, 1].set_title("Distribution de Amount (< 500, zoom)")

sns.boxplot(x="Class", y="Amount", data=df[df["Amount"] < 500], ax=axes[1, 0])
axes[1, 0].set_title("Amount par classe (zoom < 500)")
axes[1, 0].set_xticklabels(["Normale", "Fraude"])

sns.histplot(df["Time"] / 3600, bins=48, ax=axes[1, 1])
axes[1, 1].set_title("Distribution de Time (en heures écoulées)")
axes[1, 1].set_xlabel("Heures depuis la première transaction")

plt.tight_layout()
plt.show()

In [ ]:
print("Amount - transactions normales :")
print(df[df["Class"] == 0]["Amount"].describe())
print()
print("Amount - transactions frauduleuses :")
print(df[df["Class"] == 1]["Amount"].describe())

### 2.3 Corrélation des features V1-V28 avec la classe

In [ ]:
correlations = df.drop(columns=["Time"]).corr()["Class"].drop("Class").sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
correlations.plot(kind="barh", ax=ax, color=["#C44E52" if v < 0 else "#4C72B0" for v in correlations])
ax.set_title("Corrélation de chaque variable avec Class")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("Top 5 corrélations positives :")
print(correlations.tail(5))
print()
print("Top 5 corrélations négatives :")
print(correlations.head(5))

**Remarque** : certaines variables (souvent `V17`, `V14`, `V12`, `V10`, `V4`, `V11`... selon la version du dataset) ressortent avec une corrélation nettement plus marquée avec la fraude — logique, puisque le PCA d'origine a été calculé sur ces données et capture une partie du signal discriminant.

## 3. Préparation des données

- **Split stratifié** obligatoire : un split aléatoire simple pourrait, par malchance, quasiment vider le jeu de test de fraudes (il y en a si peu).
- **Scaling** : les `V1`-`V28` sont déjà standardisées par le PCA d'origine. Seules `Amount` et `Time` ont une échelle "brute" à corriger.


In [ ]:
df_model = df.copy()

scaler_amount = StandardScaler()
scaler_time = StandardScaler()
df_model["Amount_scaled"] = scaler_amount.fit_transform(df_model[["Amount"]])
df_model["Time_scaled"] = scaler_time.fit_transform(df_model[["Time"]])
df_model = df_model.drop(columns=["Amount", "Time"])

X = df_model.drop(columns=["Class"])
y = df_model["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Taux de fraude - train :", round(y_train.mean() * 100, 4), "%")
print("Taux de fraude - test  :", round(y_test.mean() * 100, 4), "%")
print("Taille train :", X_train.shape, "| Taille test :", X_test.shape)

## 4. Modèle de référence : régression logistique

On utilise `class_weight="balanced"` : ça pénalise davantage les erreurs sur la classe minoritaire pendant l'entraînement, sans dupliquer ni supprimer de données.

In [ ]:
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr, digits=3, target_names=["Normale", "Fraude"]))
print("ROC-AUC :", round(roc_auc_score(y_test, y_proba_lr), 4))
print("PR-AUC (average precision) :", round(average_precision_score(y_test, y_proba_lr), 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=["Normale", "Fraude"], ax=axes[0], cmap="Blues")
axes[0].set_title("Matrice de confusion")

RocCurveDisplay.from_predictions(y_test, y_proba_lr, ax=axes[1])
axes[1].set_title("Courbe ROC")

PrecisionRecallDisplay.from_predictions(y_test, y_proba_lr, ax=axes[2])
axes[2].set_title("Courbe Precision-Recall")

plt.tight_layout()
plt.show()

**Pourquoi la courbe Precision-Recall est plus informative que la ROC ici** : avec un déséquilibre aussi extrême, la courbe ROC peut paraître excellente même pour un modèle médiocre (le grand nombre de vrais négatifs "écrase" l'effet des faux positifs dans le calcul du taux de faux positifs). La courbe Precision-Recall, elle, se concentre sur la capacité du modèle à bien traiter la classe minoritaire.

## 5. Comparaison de modèles

### 5.1 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", max_depth=12,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, digits=3, target_names=["Normale", "Fraude"]))
print("ROC-AUC :", round(roc_auc_score(y_test, y_proba_rf), 4))
print("PR-AUC :", round(average_precision_score(y_test, y_proba_rf), 4))

### 5.2 Importance des variables (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title("Top 15 features les plus importantes (Random Forest)")
plt.tight_layout()
plt.show()

### 5.3 Rééquilibrage avec SMOTE

SMOTE génère des exemples synthétiques de la classe minoritaire (par interpolation entre voisins existants) pour rééquilibrer l'entraînement. On l'applique **uniquement sur le train set** (jamais sur le test, sinon on fausse l'évaluation avec des données synthétiques).

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Avant SMOTE :", y_train.value_counts().to_dict())
print("Après SMOTE :", y_train_smote.value_counts().to_dict())

rf_smote = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_rf_smote = rf_smote.predict(X_test)
y_proba_rf_smote = rf_smote.predict_proba(X_test)[:, 1]

print()
print(classification_report(y_test, y_pred_rf_smote, digits=3, target_names=["Normale", "Fraude"]))
print("ROC-AUC :", round(roc_auc_score(y_test, y_proba_rf_smote), 4))
print("PR-AUC :", round(average_precision_score(y_test, y_proba_rf_smote), 4))

### 5.4 Tableau comparatif

In [ ]:
results = pd.DataFrame({
    "Modèle": ["Logistic Regression (balanced)", "Random Forest (balanced)", "Random Forest + SMOTE"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_rf_smote),
    ],
    "PR-AUC": [
        average_precision_score(y_test, y_proba_lr),
        average_precision_score(y_test, y_proba_rf),
        average_precision_score(y_test, y_proba_rf_smote),
    ],
    "F1 (classe fraude)": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf_smote),
    ],
}).round(4)

results

## 6. Choix du seuil de décision

Par défaut, un modèle de classification binaire prédit "1" si la probabilité estimée dépasse 0,5. Ce seuil n'est presque jamais optimal en cas de fort déséquilibre : selon le contexte métier (coût d'une fraude manquée vs coût d'un faux positif qui bloque une transaction légitime), on ajuste le seuil.

In [ ]:
# On retient le meilleur modèle d'après le tableau ci-dessus (à ajuster selon vos résultats réels)
best_model = rf
best_proba = y_proba_rf

precisions, recalls, thresholds = precision_recall_curve(y_test, best_proba)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions[:-1], label="Précision")
ax.plot(thresholds, recalls[:-1], label="Rappel")
ax.set_xlabel("Seuil de décision")
ax.set_ylabel("Score")
ax.set_title("Précision et rappel en fonction du seuil")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Exemple : trouver le seuil qui maximise le F1-score sur la classe fraude
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_threshold_idx]

print(f"Seuil optimal (max F1) : {best_threshold:.4f}")
print(f"Précision à ce seuil   : {precisions[best_threshold_idx]:.4f}")
print(f"Rappel à ce seuil      : {recalls[best_threshold_idx]:.4f}")

y_pred_optimal = (best_proba >= best_threshold).astype(int)
print()
print(classification_report(y_test, y_pred_optimal, digits=3, target_names=["Normale", "Fraude"]))

## 7. Sauvegarde du modèle final

On sauvegarde le modèle, les scalers, et le seuil retenu — tout ce dont l'API aura besoin pour faire des prédictions en conditions réelles.

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/fraud_model.joblib")
joblib.dump(scaler_amount, "../models/scaler_amount.joblib")
joblib.dump(scaler_time, "../models/scaler_time.joblib")

import json
with open("../models/model_config.json", "w") as f:
    json.dump({
        "model_type": type(best_model).__name__,
        "decision_threshold": float(best_threshold),
        "feature_order": list(X_train.columns),
    }, f, indent=2)

print("Modèle et artefacts sauvegardés dans ../models/")

## Prochaines étapes

- Extraire cette logique validée dans des modules Python (`backend/app/`)
- Exposer le modèle via une route API `/predict`
- Construire une page Streamlit permettant de tester une transaction (manuelle ou via upload CSV) et voir la prédiction
